# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Sai Sriya Mudigonda (smudigonda@berkeley.edu)

### Notebook Structure:

1.  **Input:** Load processed data splits from `/results/train.csv` and `/results/test.csv`.
2.  Perform **EDA** to understand the output distributions and key statistics.
3.  Perform **Feature Importance Analysis** across all trained models (Logistic Baseline, Transformer, Random Forest, XGBoost).
4.  **Input:** Import all models (`*.pkl`, `*.h5`) from `/results`.
5.  **Input:** Import all model statistics (`*.csv`) from `/results`.
6.  Generate a final **model comparison table/results**.
7.  **Output:** Save the final result comparison as `/results/final_comparison.csv`.

## 1. Notebook Imports

### 1.1 Import basic and necessary libraries:

In [ ]:
# Basic imports
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.feature_selection import mutual_info_classif
import altair as alt
import seaborn as sns
import warnings

alt.data_transformers.enable("vegafusion")


# Compress all warnings
warnings.filterwarnings("ignore")

### 1.2 Import datasets:

In [ ]:
# Training dataset
df_train = pd.read_csv("../results/train.csv")
X_train = df_train.drop(columns=["readmitted"], axis=1)
Y_train = df_train["readmitted"]
# Validation dataset
df_val = pd.read_csv("../results/val.csv")
X_val = df_val.drop(columns=["readmitted"], axis=1)
Y_val = df_val["readmitted"]
# Testing dataset
df_test = pd.read_csv("../results/test.csv")
X_test = df_test.drop(columns=["readmitted"], axis=1)
Y_test = df_test["readmitted"]

In [ ]:
# Display top 5 rows of the training dataset
display(df_train.head())

In [ ]:
# Check the shape of all variables
print(f"The shape of X_train is: {X_train.shape}")
print(f"The shape of Y_train is: {Y_train.shape}")
print(f"The shape of X_val is: {X_val.shape}")
print(f"The shape of Y_val is: {Y_val.shape}")
print(f"The shape of X_test is: {X_test.shape}")
print(f"The shape of Y_test is: {Y_test.shape}")

## 2. EDA

In [ ]:
print(f"Number of columns in X_train: {len(X_train.columns)}")
print(f"List of columns in X_train: {X_train.columns.tolist()}")

In [ ]:
# combine features and target from training set
# visualize how features relate to target variable
train_combined = X_train.copy()
train_combined["readmitted"] = Y_train

train_combined["readmitted"].value_counts(normalize=True)

#### Continuous features

In [ ]:
continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
]


def plot_distributions_altair_with_kde(df: pd.DataFrame, numeric_cols):
    """
    Generates histograms with a KDE overlay and box plots side-by-side.
    """
    charts = []
    for col in numeric_cols:
        is_discrete_int = (
            pd.api.types.is_integer_dtype(df[col]) and df[col].nunique() < 30
        )
        x_binning = alt.Bin(step=1) if is_discrete_int else alt.Bin(maxbins=30)

        base = alt.Chart(df).properties(height=200)

        hist = (
            base.mark_bar(opacity=0.6)
            .encode(
                alt.X(
                    f"{col}:Q",
                    bin=x_binning,
                    title=col,
                    axis=alt.Axis(format="d", labelAngle=0),
                ),
                alt.Y("count():Q", title="Count", axis=alt.Axis(titleColor="#1f77b4")),
            )
            .properties(width=350)
        )

        density = (
            base.transform_density(col, as_=[col, "density"])
            .mark_line(color="orange", strokeWidth=3)
            .encode(
                x=f"{col}:Q",
                y=alt.Y(
                    "density:Q", axis=alt.Axis(title="Density", titleColor="orange")
                ),
            )
        )

        distribution_plot = (
            alt.layer(hist, density)
            .resolve_scale(y="independent")
            .properties(title=f"Distribution of {col}")
        )

        boxplot = (
            alt.Chart(df)
            .mark_boxplot()
            .encode(
                alt.X("readmitted:N", title="Readmitted"), alt.Y(f"{col}:Q", title=col)
            )
            .properties(title=f"{col} by Readmission", width=200, height=200)
        )

        combined_chart = distribution_plot | boxplot
        charts.append(combined_chart)

    final_chart = alt.vconcat(*charts)

    return final_chart


final_chart = plot_distributions_altair_with_kde(df_train, continuous_features)
final_chart.display()

#### Categorical features

In [ ]:
# distribution of key categorical features

categorical_features = {
    "race": [col for col in train_combined.columns if col.startswith("race_")],
    "gender": [col for col in train_combined.columns if col.startswith("gender_")],
    #'age': [col for col in train_combined.columns if col.startswith('age_')]
}

for categ_name, cols in categorical_features.items():
    plt.figure(figsize=(8, 4))
    counts = train_combined[cols].sum().sort_values(ascending=False)
    sns.barplot(x=counts.index, y=counts.values)
    plt.title(f"Distribution of {categ_name}")
    # plt.xlabel('Category')
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

#### Binary features

In [ ]:
# distribution of binary features
binary_features = ["change", "diabetesMed"]

for col in binary_features:
    plt.figure(
        figsize=(
            4,
            4,
        )
    )
    sns.countplot(data=train_combined, x=col)
    plt.title(f"Distribution of {col}")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

#### Heatmap

In [ ]:
# correlation between continuous features

plt.figure(figsize=(8, 6))
sns.heatmap(train_combined[continuous_features].corr(), annot=True, fmt=".2f")
plt.title("Correlation Matrix of Continuous Features")

#### Mutual Information Scores

In [ ]:
# correlation between features and target
# point-biseral correlation

X = train_combined.drop(columns=["readmitted"])
y = train_combined["readmitted"]

mutual_info_scores = mutual_info_classif(X, y, random_state=1234)
mutual_info_df = pd.DataFrame(
    {"Feature": X.columns, "Mutual Info Score": mutual_info_scores}
)
mutual_info_df = mutual_info_df.sort_values(by="Mutual Info Score", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=mutual_info_df.head(15), x="Mutual Info Score", y="Feature")
plt.title("Top Features by Mutual Information with Readmission")
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 3. Feature Importance Analysis

In [ ]:
# model_bsl =
# model_rf =
# model_xgb =
# model_tran =

## 4. Notebook Exports

In [ ]:
# Final Comparison